# What is tool in the context of agents

## Tool definition

Example-

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root")
def tool1(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

In [ ]:
square_root.invoke({"x": 467})

## Adding to agents

In [4]:
from langchain.agents import create_agent

from langchain.chat_models import init_chat_model

local_model = init_chat_model(
    model="local-llm",
    base_url="http://localhost:8080/v1",
    api_key="dummy",
    model_provider="openai",
    temperature=0.7)

agent = create_agent(
    model = local_model,
    tools=[tool1],
    system_prompt = "You are a specialised agent in software engineering"
)

In [5]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately 21.61.


In [6]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='b4c4134a-217b-40a4-a4cb-74e0f8635bcd'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 79, 'total_tokens': 217, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 73, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e4b-it-qat.gguf', 'system_fingerprint': 'b11146-7fe450e19', 'id': 'chatcmpl-SBv6nCOGNOGhs7p0cvwRavstNI9I22V3', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a15cff-9ac1-71d1-a078-20dfd81d4380-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'qhQ3be39BoRGInsTpwxZhbr4MoqmNf8p', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 79, 'output_tokens': 138, 'total_tokens': 217, 'input_token_detail

In [ ]:
print(response["messages"][1].tool_calls)

In [7]:
## Now lets ask the agent what is current time

response = agent.invoke(
    {"messages": [HumanMessage("what is current time")]}
    )

## print the message
pprint(response['messages'])

[HumanMessage(content='what is current time', additional_kwargs={}, response_metadata={}, id='79d24759-41db-478c-8c62-c35e9e7a890c'),
 AIMessage(content='I do not have access to real-time information, so I cannot tell you the current time.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 72, 'total_tokens': 136, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 59, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e4b-it-qat.gguf', 'system_fingerprint': 'b11146-7fe450e19', 'id': 'chatcmpl-OLdbb8LAwzeHrnvu92z2fQs9vrVGE9J6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a15d01-12d7-78e3-a288-568abbeaac91-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 64, 'total_tokens': 136, 'input_token_details': {'cache_read': 59}, 'output_token_de

In [8]:
## Create tool to get current time
from langchain.tools import tool
from datetime import datetime
@tool("get_current_date_and_time", description="Gives the current date and time")
def get_current_date_and_time():
    return datetime.now()

In [9]:
## Add the tool to agent

agent = create_agent(
    model=local_model,
    tools=[get_current_date_and_time,square_root]
)

In [10]:
## Now ask agent to get current time.
response = agent.invoke(
    {"messages": [HumanMessage("what is the current time?")]}
)
pprint(response["messages"][-1])

AIMessage(content='The current time is 11:19:10 AM on October 21, 2026.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 159, 'total_tokens': 186, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 95, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': './models/gemma4-e4b-it-qat.gguf', 'system_fingerprint': 'b11146-7fe450e19', 'id': 'chatcmpl-DrDJDC3sVmvpZV2Ytqsgp8kIXwPNfxky', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a15d01-a3a8-7a92-8f91-aa097f2548c0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 27, 'total_tokens': 186, 'input_token_details': {'cache_read': 95}, 'output_token_details': {}})


In [ ]:
## See what are all tool calls it made
print("## ====== Response ====== ## \n")
pprint(response)
## tool message
print("## ====== First AI mesage ====== ## \n")
pprint(response["messages"][1].content)
print("## ====== Tool calls ====== ## \n")
pprint(response["messages"][1].tool_calls)

In [27]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the current chief minister of TamilNadu?")

agent = create_agent(
    model=local_model
)

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The current Chief Minister of Tamil Nadu is **M. K. Stalin**.


In [25]:
# Web search tool by TavilyClient

from typing import Any, Dict
from langchain.tools import tool
from tavily import TavilyClient

from dotenv import load_dotenv
load_dotenv()

tavily_client = TavilyClient()

@tool()
def search_web(query: str) -> Dict[str, Any]:
    """Search the web for information"""

    results = tavily_client.search(query)

    return results

In [26]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the chief minister of TamilNadu?")

agent = create_agent(
    model=local_model,
    tools=[search_web]
)

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Based on the search results provided, **Thiru C. Joseph Vijay** is the Chief Minister of Tamil Nadu. 

The search results indicate that he has served in this role since May 2026.
